# paper-translator: Colab サーバー

GPU ランタイムで実行する。VS Code の Colab 拡張機能でカーネルに Colab（L4 / A100）を選んでも、ブラウザの Colab でも動く。
カーネルは Colab のマシン上で動くため、コードは GitHub から取得する（手元の変更は push → 取得セルを再実行 → カーネル再起動）。

画面と API を起動して cloudflared で公開し、最後に表示される「アプリを開く」のリンクを開く（API URL は自動で入る）。

**GPU の選び方**: モデルカードは BF16 対応 GPU（L4 / A100 など）を前提にしている。T4 は BF16 非対応で、fp16 で動くかは未確認。

| モデル | BF16 の重みサイズ |
|---|---|
| `pfnet/plamo-3-translate-2b`（既定） | 約 5.8 GB |
| `pfnet/plamo-3-translate-8b` | 約 17.1 GB |


In [1]:
!nvidia-smi

Sun Nov 22 08:18:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os

REPO = "/content/paper-translator"
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/Kame09-M/paper-translator.git {REPO}
%cd {REPO}/backend
!git -C {REPO} fetch -q origin && git -C {REPO} checkout -q feat/yamada-ui-redesign && git -C {REPO} pull -q
!git log --oneline -1
!pip install -q -U -r requirements-gpu.txt

# 注: pull で取り込んだコードの変更は、カーネルを再起動してから反映される
# （Colab の IPython の autoreload は Python 3.13 で動かない: No module named 'imp'）

/content/paper-translator/backend
553b902 (HEAD -> feat/yamada-ui-redesign, origin/feat/yamada-ui-redesign) 画面を Paper Reader のデザインに作り直す
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.68.1 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
c

## 画面をビルド

画面（frontend）をビルドして、API サーバーから一緒に配信できるようにする（1〜2分）。
Node.js は取得セルの pip（`nodejs-wheel`）で入る。


In [3]:
!cd ../frontend && npm ci --no-audit --no-fund --loglevel=error && npm run build

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 178 packages in 9s
⠼
> frontend@0.0.0 build
> tsc -b && vite build

vite v6.4.6 building for production...
✓ 39 modules transformed.
dist/index.html                              0.40 kB │ gzip:   0.27 kB
dist/assets/pdf.worker.min-CIqJB0yW.mjs  1,271.92 kB
dist/assets/index-51o9wHpw.css              11.88 kB │ gzip:   3.45 kB
dist/assets/index-C9P-0Yi0.js              749.02 kB │ gzip: 228.78 kB

(!) Some chunks are larger than 500 kB after minification. Consider:
- Using dynamic import() to code-split the application
- Use build.rollupOptions.output.manualChunks to improve chunking: https://rollupjs.org/configuration-options/#output-manualchunks
- Adjust chunk size limit for this warning via build.chunkSizeWarningLimit.
✓ built in 3.57s
⠙

## 設定

Gemini の API キーは、アプリの画面上部の「Gemini API キー」欄に入れる（ブラウザに保存され、要約・用語説明のときだけサーバーに送られる）。
ノートブックには書かない。画面で空のままなら、用語説明と要約はダミーで動く。


In [4]:
import os

os.environ["MOCK"] = "0"  # 翻訳を本物にする
os.environ["PLAMO_MODEL"] = "pfnet/plamo-3-translate-2b"

## 翻訳だけ先に試す

モデルの読み込み（初回はダウンロードあり）と、履歴（直前の段落）付きの翻訳を確かめる。


In [5]:
from app.services import translate

paras = [
    "The Transformer relies entirely on attention to draw global dependencies.",
    "Self-attention relates different positions of a single sequence.",
]
for src, tgt in zip(paras, translate.translate_paragraphs(paras, "ja")):
    print(src, "\n→", tgt, "\n")

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

modeling_plamo.py:   0%|          | 0.00/45.4k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/pfnet/plamo-3-translate-2b:
- modeling_plamo.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

tokenization_plamo.py:   0%|          | 0.00/22.4k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/pfnet/plamo-3-translate-2b:
- tokenization_plamo.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


... (6 outputs left out)

Loading weights:   0%|          | 0/243 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

[transformers] `past_key_values`/`cache_params` are deprecated. Use `cache` instead.


The Transformer relies entirely on attention to draw global dependencies. 
→ TransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransformerTransforme

## 実物の論文で確かめる（時間・要約・文脈の効果）

公開論文「Attention Is All You Need」で、要約（Gemini）と、先頭 N 段落の翻訳時間を測る。
直前の段落（文脈）あり／なしの訳も並べるので、目で見比べる（簡易評価）。


In [6]:
import time
import urllib.request

from app.services import pdf, summarize

urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", "/content/attention.pdf")
paragraphs = pdf.extract_paragraphs(open("/content/attention.pdf", "rb").read())
print("段落数:", len(paragraphs))

t = time.time()
print(summarize.summarize_paper(paragraphs, "ja"))
print(f"要約: {time.time() - t:.1f} 秒")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


段落数: 82
[研究背景]
- （ダミー）Provided proper attribution is provided,…
[研究目的]
- （ダミー）Provided proper attribution is provided,…
[方法]
- （ダミー）Provided proper attribution is provided,…
[結果]
- （ダミー）Provided proper attribution is provided,…
[考察]
- （ダミー）Provided proper attribution is provided,…
[結論]
- （ダミー）Provided proper attribution is provided,…
要約: 0.0 秒


In [7]:
N = 5
sample = paragraphs[4 : 4 + N]  # 冒頭の注記を飛ばす

t = time.time()
with_context = translate.translate_paragraphs(sample, "ja")
sec = (time.time() - t) / N
print(f"1段落あたり {sec:.1f} 秒 → 全 {len(paragraphs)} 段落なら約 {sec * len(paragraphs) / 60:.1f} 分")

without_context = [translate.translate_paragraph(p, "ja") for p in sample]

for src, a, b in zip(sample, with_context, without_context):
    print("原文:", src[:200], "\n文脈あり:", a, "\n文脈なし:", b, "\n")

1段落あたり 80.2 秒 → 全 82 段落なら約 109.6 分
原文: 31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA. 
文脈あり: 第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第第

## API サーバー起動

起動後、Colab の中から1回翻訳を呼んでモデルを読み込んでおく（初回はダウンロード込みで数分）。
読み込まずに公開すると、画面からの最初のリクエストがモデル読み込みを待って cloudflared の時間制限（約100秒）で切れるため。


In [8]:
import json
import subprocess
import time
import urllib.request

server = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("server.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(60):  # 起動待ち
    try:
        print(urllib.request.urlopen("http://localhost:8000/api/health").read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    print(open("server.log").read())
    raise RuntimeError("サーバーが起動しない（上のログを確認）")

t = time.time()
req = urllib.request.Request(
    "http://localhost:8000/api/translate-paragraph",
    json.dumps({"text": "Hello.", "target_lang": "ja"}).encode(),
    {"Content-Type": "application/json"},
)
print(json.load(urllib.request.urlopen(req, timeout=1800)), f"モデル読み込み込み {time.time() - t:.0f} 秒")

{"status":"ok","mock":false}
{'translation': 'こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。こんにちは。

## cloudflared で公開

最後に出る「アプリを開く」のリンクを開く。画面も API もこの URL から配信されるので、API URL 欄は自動で入る。
cloudflared は裏で動き続ける（カーネルを止めると止まる）。リンクを開いてすぐはつながらないことがあるので、数秒待って再読み込みする。


In [9]:
import re
import subprocess
import time

from IPython.display import Markdown, display

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

# 裏で動かし、ログから公開 URL を拾う（ログはファイルに書くので、出力が詰まって止まることがない）
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=open("cloudflared.log", "w"),
    stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    found = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if found:
        url = found.group(0)
        break
    time.sleep(1)
if url is None:
    print(open("cloudflared.log").read())
    raise RuntimeError("cloudflared の URL が出ない（上のログを確認）")
display(Markdown(f"**アプリを開く**: [{url}]({url})"))

**アプリを開く**: [https://hosted-bronze-formed-miracle.trycloudflare.com](https://hosted-bronze-formed-miracle.trycloudflare.com)